Mixed Training - pseudo-labels



Inputs:
- data/augmented_training_data.csv
- data/pseudo_labeled_chunks.csv
- models/roberta-dapt-dynamic

Outputs:
- models/roberta-mixed-final/
- results/mixed_training_metrics.json


In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import pickle
import warnings
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import torch.nn as nn

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("✅ Libraries loaded")
print(f"Device: {device}")

✅ Libraries loaded
Device: cuda


## 1) Load real and pseudo-labeled data

In [2]:
print("Loading real augmented data...")
real_df = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])
real_train = real_df[real_df['split']=='train'].copy()
real_val = real_df[real_df['split']=='validation'].copy()
real_test = real_df[real_df['split']=='test'].copy()
for d in (real_train, real_val, real_test):
    d['sample_weight'] = 1.0
print(f"Real train: {len(real_train):,}, val: {len(real_val):,}, test: {len(real_test):,}")

print("\nLoading pseudo-labeled data...")
pseudo = pd.read_csv('data/pseudo_labeled_chunks.csv', keep_default_na=False, na_values=[''])
pseudo.rename(columns={'pseudo_label':'frame_label'}, inplace=True)
from sklearn.model_selection import train_test_split
pseudo_train, pseudo_val = train_test_split(
    pseudo, test_size=0.10, random_state=42, stratify=pseudo['frame_label']
)
pseudo_train['sample_weight'] = 0.2
pseudo_val['sample_weight'] = 0.2
print(f"Pseudo train: {len(pseudo_train):,}, val: {len(pseudo_val):,}")

combined_train = pd.concat([
    real_train[['chunk_text','frame_label','sample_weight']],
    pseudo_train[['chunk_text','frame_label','sample_weight']]
], ignore_index=True)

print(f"\n✅ Combined training: {len(combined_train):,} samples")
print(f"✅ Real validation: {len(real_val):,} samples (for early stopping)")
print(f"✅ Real test: {len(real_test):,} samples")

Loading real augmented data...
Real train: 4,606, val: 683, test: 348

Loading pseudo-labeled data...
Pseudo train: 23,684, val: 2,632

✅ Combined training: 28,290 samples
✅ Real validation: 683 samples (for early stopping)
✅ Real test: 348 samples


## 2) Encode labels and tokenize

In [3]:
with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
labels = list(label_encoder.classes_)
num_labels = len(labels)

# Encode labels
combined_train['label'] = label_encoder.transform(combined_train['frame_label'])
real_val['label'] = label_encoder.transform(real_val['frame_label'])
real_test['label'] = label_encoder.transform(real_test['frame_label'])

# Class weights from real training data only
orig_real = real_train.copy()
orig_real['label'] = label_encoder.transform(orig_real['frame_label'])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(orig_real['label']),
    y=orig_real['label']
)
class_weights = torch.FloatTensor(class_weights)
print(f"✅ Class weights computed: {class_weights}")

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained('models/roberta-dapt-dynamic')
print("✅ Tokenizer loaded from dynamic model")

def tokenize_function(examples):
    return tokenizer(
        examples['chunk_text'], truncation=True, max_length=384, padding='max_length'
    )

# Create datasets
train_dataset = Dataset.from_pandas(combined_train[['chunk_text','label','sample_weight']])
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
train_dataset.set_format('torch')

# FIX 1: REAL validation only for early stopping
val_dataset = Dataset.from_pandas(real_val[['chunk_text','label']])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
val_dataset.set_format('torch')

test_dataset = Dataset.from_pandas(real_test[['chunk_text','label']])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
test_dataset.set_format('torch')

print(f"✅ Train: {len(train_dataset)}, Val (REAL only): {len(val_dataset)}, Test: {len(test_dataset)}")

✅ Class weights computed: tensor([0.8246, 1.1883, 1.4706, 0.8149, 1.2127, 0.8237])
✅ Tokenizer loaded from dynamic model


Map:   0%|          | 0/28290 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

✅ Train: 28290, Val (REAL only): 683, Test: 348


## 3) FIX 5: Dynamic Class Weighting System

In [4]:
class DynamicClassWeightCallback(TrainerCallback):
    """Adjust class weights every 50 steps AND log comprehensive metrics"""
    def __init__(self, trainer, num_classes, initial_weights, labels):
        self.trainer = trainer
        self.num_classes = num_classes
        self.class_weights = initial_weights.clone()
        self.labels = labels
        self.update_freq = 50
        
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.update_freq == 0 and state.global_step > 0:
            model = self.trainer.model
            model.eval()
            
            # Get train loss from log history
            train_loss = None
            if state.log_history:
                for log_entry in reversed(state.log_history):
                    if 'loss' in log_entry:
                        train_loss = log_entry['loss']
                        break
            
            # Get current learning rate
            current_lr = None
            if hasattr(self.trainer, 'optimizer'):
                current_lr = self.trainer.optimizer.param_groups[0]['lr']
            
            # Compute validation metrics (loss + per-class F1)
            val_preds = []
            val_labels = []
            val_losses = []
            loss_fct = nn.CrossEntropyLoss(reduction='mean')
            
            with torch.no_grad():
                for batch in self.trainer.get_eval_dataloader():
                    batch = {k: v.to(args.device) for k, v in batch.items()}
                    labels = batch.pop('labels')
                    outputs = model(**batch)
                    
                    # Compute loss
                    loss = loss_fct(outputs.logits, labels)
                    val_losses.append(loss.item())
                    
                    # Get predictions
                    preds = outputs.logits.argmax(dim=-1)
                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
            
            # Compute per-class F1
            val_preds = np.array(val_preds)
            val_labels = np.array(val_labels)
            _, _, f1_per_class, _ = precision_recall_fscore_support(
                val_labels, val_preds, average=None, zero_division=0.0
            )
            macro_f1 = np.mean(f1_per_class)
            val_loss = np.mean(val_losses)
            
            # Update weights: inverse F1 (lower F1 = higher weight)
            epsilon = 0.1
            inverse_f1 = [1.0 / (f1 + epsilon) for f1 in f1_per_class]
            mean_inv = sum(inverse_f1) / len(inverse_f1)
            new_weights = torch.FloatTensor([w / mean_inv for w in inverse_f1])
            
            # Smooth blend: 80% old + 20% new
            self.class_weights = 0.8 * self.class_weights + 0.2 * new_weights
            
            # Clamp to [0.5, 2.0] range
            self.class_weights = torch.clamp(self.class_weights, 0.5, 2.0)
            
            # Update trainer's weights
            if hasattr(self.trainer, 'class_weights'):
                self.trainer.class_weights = self.class_weights.to(args.device)
            
            # ============================================================
            # ENHANCED OUTPUT: Comprehensive Metrics Table
            # ============================================================
            print("\n" + "="*80)
            print(f"📊 STEP {state.global_step} METRICS SUMMARY (Epoch {state.epoch:.2f})")
            print("="*80)
            
            # Loss metrics
            if train_loss is not None:
                print(f"  Train Loss      : {train_loss:.4f}")
            print(f"  Val Loss        : {val_loss:.4f}")
            
            # Learning rate
            if current_lr is not None:
                print(f"  Learning Rate   : {current_lr:.6e}")
            
            # Macro F1
            print(f"  Macro F1 (Val)  : {macro_f1:.4f}")
            
            print("-"*80)
            print("  PER-CLASS F1 SCORES & DYNAMIC WEIGHTS:")
            print("-"*80)
            
            # Per-class metrics in table format
            for label, weight, f1 in zip(self.labels, self.class_weights, f1_per_class):
                bar_length = int(f1 * 40)  # Progress bar (max 40 chars)
                bar = "█" * bar_length + "░" * (40 - bar_length)
                print(f"  {label:20s}: F1={f1:.3f} {bar} [w={weight:.3f}]")
            
            print("="*80 + "\n")
            
            model.train()

print("✅ Enhanced dynamic callback with comprehensive metrics logging defined")

✅ Enhanced dynamic callback with comprehensive metrics logging defined


## 4) FIX 3: Custom Data Collator + Weighted CE Trainer

In [5]:
from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import DataCollatorWithPadding
import torch.nn as nn

@dataclass
class DataCollatorWithSampleWeight(DataCollatorWithPadding):
    """Collator that preserves sample_weight field"""
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        sample_weights = None
        if 'sample_weight' in features[0]:
            sample_weights = torch.FloatTensor([f.pop('sample_weight') for f in features])
        
        batch = super().__call__(features)
        
        if sample_weights is not None:
            batch['sample_weight'] = sample_weights
        
        return batch

class DynamicWeightedCETrainer(Trainer):
    """Trainer with Weighted Cross-Entropy + dynamic class weights"""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        sample_weight = inputs.pop('sample_weight', None)
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # FIX 3: Use standard Cross-Entropy with class weights
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights,
            reduction='none'
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        # Apply sample weighting if present (training)
        if sample_weight is not None:
            weighted = (loss * sample_weight).mean()
        else:
            weighted = loss.mean()
        
        return (weighted, outputs) if return_outputs else weighted

print("✅ Weighted CE Trainer with dynamic weights defined")

✅ Weighted CE Trainer with dynamic weights defined


## 5) Load model with dynamic dropout settings

In [6]:
print("Loading dynamic model from models/roberta-dapt-dynamic...")
model = RobertaForSequenceClassification.from_pretrained(
    'models/roberta-dapt-dynamic',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.05,  # From dynamic model
    hidden_dropout_prob=0.10,  # From dynamic model
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("✅ Model loaded with dynamic settings")

Loading dynamic model from models/roberta-dapt-dynamic...
✅ Model loaded with dynamic settings


## 6) FIX 2: Differential Learning Rates

In [7]:
# Separate encoder and classifier parameters
encoder_params = []
classifier_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        classifier_params.append(param)
    else:
        encoder_params.append(param)

# FIX 2: Differential LR (encoder: 2e-5, classifier: 1e-3)
optimizer = AdamW([
    {'params': encoder_params, 'lr': 2e-5},      # Pretrained encoder
    {'params': classifier_params, 'lr': 1e-3}    # New classifier (50x faster!)
], weight_decay=0.05)

print("✅ Optimizer configured with differential LRs:")
print("   Encoder: 2e-5")
print("   Classifier: 1e-3 (50x faster)")

✅ Optimizer configured with differential LRs:
   Encoder: 2e-5
   Classifier: 1e-3 (50x faster)


## 7) FIX 4: Optimized Training Arguments

In [8]:
# Calculate training steps for scheduler
num_training_steps = (len(train_dataset) // (8 * 4)) * 15  # batch * grad_accum * epochs
num_warmup_steps = int(0.15 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f"Training schedule: {num_training_steps} steps, {num_warmup_steps} warmup")

# FIX 4: Optimized hyperparameters
training_args = TrainingArguments(
    output_dir='models/roberta-mixed-final',
    num_train_epochs=15,  # More epochs
    warmup_ratio=0.15,
    per_device_train_batch_size=8,  # Smaller batches
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,  # More accumulation
    weight_decay=0.05,
    max_grad_norm=5.0,  # More relaxed
    eval_strategy='epoch',  # Evaluate per epoch
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    label_smoothing_factor=0.0,  # No label smoothing (pseudo-labels are soft)
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42,
)

print("✅ Training arguments configured")
print(f"   Effective batch size: {8 * 4} (8 batch × 4 accum)")
print(f"   Total epochs: 15")
print(f"   Early stopping patience: 5")

Training schedule: 13260 steps, 1989 warmup
✅ Training arguments configured
   Effective batch size: 32 (8 batch × 4 accum)
   Total epochs: 15
   Early stopping patience: 5


## 8) Metrics function

In [9]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    
    # Per-class F1 for monitoring
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels_np, preds, average=None, zero_division=0
    )
    
    metrics = {
        'accuracy': float(acc),
        'f1_macro': float(f1_macro)
    }
    
    for idx, label_name in enumerate(labels):
        metrics[f'f1_{label_name}'] = float(f1_per_class[idx])
    
    return metrics

print("✅ Metrics function defined")

✅ Metrics function defined


## 9) Initialize Trainer and Train

In [10]:
# Initialize custom collator
data_collator = DataCollatorWithSampleWeight(tokenizer=tokenizer, padding=True)

# Initialize dynamic callback
dynamic_callback = DynamicClassWeightCallback(
    trainer=None,  # Will be set after trainer creation
    num_classes=num_labels,
    initial_weights=class_weights,
    labels=labels
)

# Initialize trainer
trainer = DynamicWeightedCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # FIX 1: REAL validation only
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),  # More patience
        dynamic_callback
    ],
    class_weights=class_weights,
    data_collator=data_collator,
    optimizers=(optimizer, scheduler)  # FIX 2: Custom optimizer
)

# Link callback to trainer
dynamic_callback.trainer = trainer

print("\n🚀 Starting OPTIMIZED mixed training...")
print("="*80)
print("IMPROVEMENTS ACTIVE:")
print("  ✅ FIX 1: Real validation only for early stopping")
print("  ✅ FIX 2: Differential LR (encoder: 2e-5, classifier: 1e-3)")
print("  ✅ FIX 3: Weighted Cross-Entropy (no Focal Loss)")
print("  ✅ FIX 4: Optimized hyperparams (batch=8, accum=4, epochs=15)")
print("  ✅ FIX 5: Dynamic class weighting (adapts every 50 steps)")
print("="*80)

train_result = trainer.train()
print("\n✅ Training complete!")


🚀 Starting OPTIMIZED mixed training...
IMPROVEMENTS ACTIVE:
  ✅ FIX 1: Real validation only for early stopping
  ✅ FIX 2: Differential LR (encoder: 2e-5, classifier: 1e-3)
  ✅ FIX 3: Weighted Cross-Entropy (no Focal Loss)
  ✅ FIX 4: Optimized hyperparams (batch=8, accum=4, epochs=15)
  ✅ FIX 5: Dynamic class weighting (adapts every 50 steps)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
1,0.067300,2.677899,0.682284,0.680706,0.612245,0.831933,0.740741,0.589744,0.716981,0.592593
2,0.150300,2.363807,0.653001,0.653788,0.612245,0.784810,0.717300,0.583026,0.716418,0.508929
3,0.071800,2.476523,0.670571,0.670862,0.654709,0.791304,0.696035,0.566210,0.746667,0.570248
4,0.050600,2.640568,0.689605,0.692524,0.661290,0.800000,0.743590,0.611570,0.704545,0.634146
5,0.061400,2.748897,0.682284,0.683744,0.595122,0.783019,0.741036,0.619048,0.747573,0.616667
6,0.023000,2.857640,0.680820,0.679715,0.596859,0.794643,0.734375,0.585774,0.755760,0.610879
7,0.030700,2.852038,0.673499,0.671227,0.607843,0.801688,0.723735,0.547085,0.736842,0.610169
8,0.016900,3.135568,0.693997,0.692076,0.675799,0.809524,0.743802,0.602510,0.724490,0.596330
9,0.012700,3.180638,0.686676,0.688328,0.643902,0.815451,0.725738,0.586207,0.746269,0.612403



📊 STEP 50 METRICS SUMMARY (Epoch 0.06)
  Val Loss        : 1.3794
  Learning Rate   : 4.826546e-07
  Macro F1 (Val)  : 0.6831
--------------------------------------------------------------------------------
  PER-CLASS F1 SCORES & DYNAMIC WEIGHTS:
--------------------------------------------------------------------------------
  Conflict            : F1=0.664 ██████████████████████████░░░░░░░░░░░░░░ [w=0.862]
  Economic            : F1=0.824 ████████████████████████████████░░░░░░░░ [w=1.118]
  Human Impact        : F1=0.730 █████████████████████████████░░░░░░░░░░░ [w=1.363]
  Moral Value         : F1=0.595 ███████████████████████░░░░░░░░░░░░░░░░░ [w=0.875]
  None                : F1=0.710 ████████████████████████████░░░░░░░░░░░░ [w=1.161]
  Powerlessness       : F1=0.575 ███████████████████████░░░░░░░░░░░░░░░░░ [w=0.888]


📊 STEP 100 METRICS SUMMARY (Epoch 0.11)
  Train Loss      : 0.0422
  Val Loss        : 1.4382
  Learning Rate   : 9.854198e-07
  Macro F1 (Val)  : 0.6837
----------

## 10) Evaluate and Save

In [11]:
print("\nEvaluating on REAL validation...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nEvaluating on REAL test...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

# Save model
os.makedirs('models/roberta-mixed-final', exist_ok=True)
trainer.save_model('models/roberta-mixed-final')
tokenizer.save_pretrained('models/roberta-mixed-final')

# Save metrics
os.makedirs('results', exist_ok=True)
with open('results/mixed_training_metrics.json', 'w') as f:
    json.dump({
        'improvements': [
            'Real validation for early stopping',
            'Differential learning rates (encoder: 2e-5, classifier: 1e-3)',
            'Weighted Cross-Entropy (replaced Focal Loss)',
            'Optimized hyperparams (batch=8, accum=4, epochs=15)',
            'Dynamic class weighting'
        ],
        'real_val': val_results,
        'real_test': test_results,
        'training_time': train_result.metrics['train_runtime']
    }, f, indent=2)

print("\n✅ Model and metrics saved to models/roberta-mixed-final/")
print("\n" + "="*80)
print("FINAL RESULTS:")
print(f"  Real Validation F1: {val_results['eval_f1_macro']:.4f}")
print(f"  Real Test F1: {test_results['eval_f1_macro']:.4f}")
print("="*80)


Evaluating on REAL validation...


KeyboardInterrupt: 